# 강화학습
에이전트의 라벨이 없이 직접 행동해보고 보상을 받아 어떤 행동이 좋은지 점점(차근차근) 학습해 나가는 방식

강화 학습은 정답이 아니라(지도학습X) 보상으로 학습한다. - 진화의 개념

강화학습은 경험 기반 학습이다.

현재 상태 Q(State, Action) 에 대해 보상을 받는게 Q_value값인데
Q-value값이 좋은쪽으로 점점 진행함

현재상태 -> 행동선택 -> 보상확인 -> 다음 행동을 개선 순으로 진행.

### 순서
1. 상태는 현재의 위치
2. 행동은 위/아래/좌/우
3. 보상은 행동 결과에 대한 점수
4. Q-table은 상태별 행동 점수 표
5. Q-learning은 Q-table을 조금씩 갱신해 나감
6. epsilon-greedy는 탐험과 이용을 조절함
7. 학습 후 Q-table에서 가장 큰 행동을 선택하면 이것이 정책(policy)이 됨
8. 이 구조가 Gymnasium의 step() 구조와 연결 된다.

In [64]:
import numpy as np
import random

np.random.seed(42)
random.seed(42)

# GirdWorld 환경 선언 (3행 4열)
ROWS = 3
COLS = 4

START = (0,0) # Agent의 시작 위치
GOAL = (2,3)  # 목표
TRAP = (1,1)  # 함정

actions = {
    0:(-1, 0),  # 상 방향
    1:(1, 0),   # 하 방향
    2:(0, -1),  # 좌 방향
    3:(0, 1)    # 우 방향
}
# 사람이 이해하기 쉽게 이름 줘보기
actions_name = {
    0:'상',
    1:'하',
    2:'좌',
    3:'우',
}

# 전체 상태 가능 수
num_states = ROWS * COLS
# 가능한 행동 수
num_actions = len(actions)

# Q-table 초기화
Q = np.zeros((num_states, num_actions))
# print(Q)

# ** 하이퍼파라미터 ** <- 이해 해야함!
alpha = 0.1     # 학습률
gamma = 0.9     # 할인율 미래보상을 얼마나 중요하게 볼지
epsilon=1.0     # 초기 탐험율
epsilon_decay = 0.995 # 에피소드가 끝날때 마다 탐험율(epsilon)을 줄이는 비율
epsilon_min = 0.1     # 에피소드가 끝날때 마다 탐험율(epsilon)을 줄이는 최소 비율 = 탐험을 하지 못하게 하는걸 방지 = 0이 되는것을 방지
max_steps = 30  # 하나의 에피소드내에서 최대 이동 가능 횟수 - 30개동안 목표에 도달하지 못하거나 함정에 빠지면 멈춤.
episodes = 1000

# 테이블의 행렬 변환을 자유롭게 바꿀 수 있어야 한다
# 상태 변환 함수 : 위치 정보를 Q-tabel에서 사용할 상태 번호로 변환(0,0) -> 0 ... (2,3) -> 11
def pos_to_state(pos):
  row, col = pos
  return row * COLS + col
print(pos_to_state((2,3)))

# 상태번호를 위치로 변환 : 11 -> (2,3)
def state_to_pos(state):
  row = state // COLS
  col = state % COLS
  return (row, col)
print(state_to_pos(11))

11
(2, 3)


In [65]:
# 환경 이동 함수
# 현재 위치에서 행동을 실행하고 결과를 반환하는 함수
def step(pos, action):
  row, col = pos
  # 선택한 행동에 해당하는 행변화량, 열변화량 얻기
  dr, dc = actions[action]

  # 이동 후의 행 위치 계산
  next_row = row + dr
  # 이동 후의 열 위치 계산
  next_col = col + dc

  # GridWorld 경계 밖으로 나가면 제자리에 있게 하고 패널티를 부여함
  if next_row < 0 or next_row >= ROWS or next_col < 0 or next_col >= COLS:
    next_pos = pos
    reward = -2
    done = False # 벽에 부딪혔다고 에피소드가 끝나지는 않음
    return next_pos, reward, done

  # 벨만 방정식을 위해 필요한 값.
  next_pos = (next_row, next_col)

  # 목표지점과 함정 도착 여부 따른 보상값과 패널티 값 부여후 에피소드 끝
  if next_pos == GOAL:
    reward = 10
    done = True # 에피소드 종료
  elif next_pos == TRAP:
    reward = -10
    done = True # 에피소드 종료
  # 일반 이동할 때 마다 -1 패널티 부여 -> 짧은 경로를 유도하기 위해
  else:
    reward = -1
    done = False

  return next_pos, reward, done

print(step((0,0), 3)) # ((0, 1), -1, False)
print(step((1,0), 3)) # ((1, 1), -10, True)
print(step((2,2), 3)) # ((2, 3), 10, True)


((0, 1), -1, False)
((1, 1), -10, True)
((2, 3), 10, True)


In [68]:
# epsilon-greedy 행동 선택 : 탐험 또는 이용
def choose_action(state, epsilon):
  if random.random() < epsilon:
    return random.randint(0, num_actions-1) # 탐험
  else:
    return np.argmax(Q[state]) # 이용 : 현재상태에서 Q값이 가장 큰 행동 선택
print('선택된 행동은 ?',choose_action(0, 1.0)) # 행동이 선택 랜덤

# Q-learning 학습 시작하기
for episode in range(episodes):
  pos = START
  state = pos_to_state(pos) # 좌표(0,0) -> 상태번호:0

  # 하나의 에피소드 안에서 최대 행동 횟수
  for step_count in range(max_steps):

    # 행동 선택
    action = choose_action(state, epsilon)
    # print('action :', action)

    # 행동 실행
    next_pos, reward, done = step(pos, action)
    # print('next_pos :', next_pos)
    # print('reward :', reward)
    # print('done :', done)

    # 다음 위치 -> 상태번호로 변환
    next_state = pos_to_state(next_pos)
    # print('next_state :', next_state)

    # 현재 Q값(벨만 방정식에 적용)
    old_q = Q[state][action] # 현재 상태(state)에서 선택한 action의 기존 Q-value값
    # print('old_q :', old_q)

    # Q-learning Target 계산하기 - 이번 경험으로 계산한 목표 Q값
    # 목표에 도착했거나 함정을 만났을때 미래 Q값을 보지 않고 현재 보상만 사용
    if done:
      target = reward
    # 에피소드가 안 끝난경우
    else:
      # 새로운 정보에 기반에 기존 Q값을 업데이트 한다 - Q-learnig의 핵심!
      next_max_q = np.max(Q[next_state])    # 다음 상태에서 가능한 행동 중 가장 큰 Q값
      target = reward + gamma * next_max_q  # 목표 Q값
    # print('target :', target)

    # Q-learning update
    # 목표 값 - 기존 Q값
    td_error = target - old_q
    # Q state행에 action열
    # 현재 상태의 가치를 보상과 다음상태의 가치로 표현한 식 - 벨만 방정식
    Q[state][action] = old_q + alpha * td_error

    # 상태 이동
    pos = next_pos # 현재위치를 다음 위치로 갱신
    state = next_state # 현재 상태번호를 다음 상태번호로 갱신

    if done:
      break

  # epsilon 감소
  epsilon = max(epsilon_min, epsilon * epsilon_decay)
  # print("epsilon :",epsilon)

# 학습 결과 출력하기
print('학습된 Q-tabel :\n',np.round(Q, 2))

print('\n 각 상태에서 가장 좋은 행동 출력하기 : ')
for state in range(num_states):
  pos = state_to_pos(state)

  if pos == GOAL:
    print(f'상태:{state} {pos} : 목표지점')
    continue

  if pos == TRAP:
    print(f'상태:{state} {pos} : 함정')
    continue

  # 이 상태에서 가장 best인 action을 찾음.
  best_action = np.argmax(Q[state])
  best_action_name = actions_name[best_action]
  print(f'상태:{state} {pos} : {best_action_name}')


선택된 행동은 ? 3
학습된 Q-tabel :
 [[  0.81   3.07   0.81   3.12]
 [  2.12 -10.     1.81   4.58]
 [  3.58   6.2    3.12   6.2 ]
 [  5.2    8.     4.58   5.2 ]
 [ -0.29   4.56  -2.19  -9.69]
 [  0.     0.     0.     0.  ]
 [  2.18   3.79  -7.71   8.  ]
 [  6.2   10.     6.2    7.  ]
 [ -0.83  -0.24  -1.16   6.19]
 [ -8.33   1.84   1.15   8.  ]
 [  1.04   1.34   1.89  10.  ]
 [  0.     0.     0.     0.  ]]

 각 상태에서 가장 좋은 행동 출력하기 : 
상태:0 (0, 0) : 우
상태:1 (0, 1) : 우
상태:2 (0, 2) : 우
상태:3 (0, 3) : 하
상태:4 (1, 0) : 하
상태:5 (1, 1) : 함정
상태:6 (1, 2) : 우
상태:7 (1, 3) : 하
상태:8 (2, 0) : 우
상태:9 (2, 1) : 우
상태:10 (2, 2) : 우
상태:11 (2, 3) : 목표지점


In [80]:
# 학습된 정책으로 실제 이동경로로 확인
pos = START
path = [pos] # 시작 위치부터 이동경로를 저장할 list
for i in range(20):
  state = pos_to_state(pos)
  action = np.argmax(Q[state])
  next_pos, reward, done = step(pos, action)
  path.append(next_pos) # 이동한 다음위치를 경로에 추가

  pos = next_pos
  if done:
    break

print('이동 경로 :', path)

arrow = {
    0:'↑',
    1:'↓',
    2:'←',
    3:'→',
}
print('이동 경로 화살표로 출력:')
for r in range(ROWS):
  row_text = ''
  for c in range(COLS):
    pos = (r, c)

    if pos == START:
      row_text += ' START '
    elif pos == GOAL:
      row_text += ' GOAL~ '
    elif pos == TRAP:
      row_text += ' TRAP! '
    else:
      state = pos_to_state(pos)
      best_action = np.argmax(Q[state])
      row_text += f' {arrow[best_action]} '

  print(row_text)

이동 경로 : [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3)]
이동 경로 화살표로 출력:
 START  →  →  ↓ 
 ↓  TRAP!  →  ↓ 
 →  →  →  GOAL~ 
